# SadakAI - YOLOv8 Training Notebook

Road Hazard Detection Model Training

## 1. Setup & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content

!pip install ultralytics -q
!pip install -U albumentations -q

## 2. Upload Dataset

In [ ]:
# Upload your dataset zip file
from google.colab import files
uploaded = files.upload()

In [ ]:
# Extract dataset
!unzip -q road_hazard_dataset.zip -d model/data/
!ls -la model/data/

## 3. Training Configuration

In [ ]:
from ultralytics import YOLO
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Training parameters
EPOCHS = 100
IMG_SIZE = 640
BATCH = 16
PATIENCE = 20
MODEL_NAME = "yolov8m.pt"  # Medium model - best accuracy/speed balance

# Create data.yaml for training
data_yaml = '''
path: /content/model/data
train: images/train
val: images/val
test: images/test

nc: 4
names:
  0: pothole
  1: crack
  2: speed_breaker
  3: waterlogging
'''

with open('data.yaml', 'w') as f:
    f.write(data_yaml)

## 4. Start Training

In [ ]:
# Load model
model = YOLO(MODEL_NAME)

# Train
results = model.train(
    data='data.yaml',
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    project='runs/train',
    name='sadakai_yolov8m',
    exist_ok=True,
    
    # Augmentation
    augment=True,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    
    # Optimization
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    
    # Regularization
    dropout=0.1,
    
    # Misc
    verbose=True,
    seed=42,
)

## 5. Evaluate & Export

In [ ]:
# Evaluate on test set
!python -m ultralytics val model=runs/train/sadakai_yolov8m/weights/best.pt data=data.yaml split=test

In [ ]:
# Export to ONNX for deployment
model = YOLO('runs/train/sadakai_yolov8m/weights/best.pt')
model.export(format='onnx')

In [ ]:
# Download weights
import shutil
shutil.copy('runs/train/sadakai_yolov8m/weights/best.pt', '/content/drive/MyDrive/SadakAI/best.pt')
shutil.copy('runs/train/sadakai_yolov8m/weights/best.onnx', '/content/drive/MyDrive/SadakAI/best.onnx')

print("Model saved to Google Drive!")

## 6. Test Inference

In [ ]:
# Test inference
from PIL import Image

model = YOLO('runs/train/sadakai_yolov8m/weights/best.pt')

# Run inference
results = model.predict(
    source='/content/model/data/images/val/',
    conf=0.25,
    iou=0.45,
    save=True
)

print("Inference complete!")